# Analisis Exploratorio de Datos - Reto Alkomprar


**Autor: Juan Esteban Henao Palacio**  
**Fecha: 26 Abril del 2026**  

---

In [32]:
import pandas as pd
import os
import sys
sys.path.append("../src")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from datetime import datetime
from functions import apply_theme, title_card


In [33]:
excel_path = '../data/Prueba_Tecnica_Base_BI.xlsx'

excel_file = pd.ExcelFile(excel_path)
#Hojas disponibles
df_orders = pd.read_excel(excel_path, sheet_name='Ordenes')
df_products = pd.read_excel(excel_path, sheet_name='Producto')
df_regions = pd.read_excel(excel_path, sheet_name='Region')

print(f"\nInformacion de los dataframes:")
print(f"  - Ordenes: {df_orders.info()}")
print(f"  - Producto: {df_products.info()}")
print(f"  - Region: {df_regions.info()}")


Informacion de los dataframes:
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_orden             51290 non-null  str           
 1   Fecha_de_la_orden    51290 non-null  datetime64[us]
 2   Fecha_de_envío       51290 non-null  datetime64[us]
 3   Método_de_envío      51290 non-null  str           
 4   Nombre_del_cliente   51262 non-null  str           
 5   Estado_Departamento  51290 non-null  str           
 6   País                 51290 non-null  str           
 7   ID_producto          51290 non-null  str           
 8   Ventas               51289 non-null  float64       
 9   Cantidad             51289 non-null  float64       
 10  Descuento            51289 non-null  float64       
 11  Ganancia             51289 non-null  float64       
 12  Costo_de_envío       51289 non-null  float64       
 13  Prioridad_

In [34]:
print("Informacion hoja de Ordenes:")
print(f"\nMemoria usada: {df_orders.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df_orders.describe(include='all')

Informacion hoja de Ordenes:



Memoria usada: 23.43 MB


,ID_orden,Fecha_de_la_orden,Fecha_de_envío,Método_de_envío,Nombre_del_cliente,Estado_Departamento,País,ID_producto,Ventas,Cantidad,Descuento,Ganancia,Costo_de_envío,Prioridad_de_orden
count,51290,51290,51290,51290,51262,51290,51290,51290,51289.000000,51289.000000,51289.000000,51289.000000,51289.000000,51290
unique,25035,NaN,NaN,4,795,1094,148,10292,NaN,NaN,NaN,NaN,NaN,4
top,CA-2014-100111,NaN,NaN,Standard Class,Muhammed Yedwab,California,United States,OFF-AR-10003651,NaN,NaN,NaN,NaN,NaN,Medium
freq,14,NaN,NaN,30775,108,2001,9994,35,NaN,NaN,NaN,NaN,NaN,29433
mean,NaN,2023-05-12 12:56:52.875804,2023-05-16 12:15:28.180932,NaN,NaN,NaN,NaN,NaN,246.494422,3.476496,0.142910,28.642165,26.376290,NaN
min,NaN,2021-01-01 00:00:00,2021-01-03 00:00:00,NaN,NaN,NaN,NaN,NaN,0.444000,1.000000,0.000000,-6599.978000,0.002000,NaN
25%,NaN,2022-06-19 00:00:00,2022-06-23 00:00:00,NaN,NaN,NaN,NaN,NaN,30.756000,2.000000,0.000000,0.000000,2.610000,NaN
50%,NaN,2023-07-08 00:00:00,2023-07-12 00:00:00,NaN,NaN,NaN,NaN,NaN,85.056000,3.000000,0.000000,9.240000,7.790000,NaN
75%,NaN,2024-05-22 00:00:00,2024-05-26 00:00:00,NaN,NaN,NaN,NaN,NaN,251.067600,5.000000,0.200000,36.810000,24.450000,NaN
max,NaN,2034-12-08 00:00:00,2034-12-12 00:00:00,NaN,NaN,NaN,NaN,NaN,22638.480000,14.000000,0.850000,8399.976000,933.570000,NaN


#### NOTAS Hoja de Ordenes

Existen **ID_orden** duplicados, esto es porque un pedido puede contener varios productos.

Se observa que existen 28 registros de **Nombre_del_cliente** vacios o nulos, esta variable se imputara con el valor de "Cliente Desconocido" para mantener la integridad de los datos.

Existe 1 registro con los campos de **Ventas, Cantidad, Descuento, Ganancia, Costo_de_envío** Vacios o nulos, este registro se eliminara ya que no aporta informacion relevante para el analisis y podria afectar los resultados de los modelos predictivos.

Se identificaron varios regitros con Paìs registrado: **Colombia-** este error de digitacion, se va a corregir a Colombia para mantener la consistencia de los datos.

Se identifico un año erroneo [2034] en la columna de Fecha_de_la_orden, este registro se corregira a 2024 para mantener la consistencia de los datos.

La columna ventas tiene un dato inconsistente, con valor de 0, este registro se eliminara por ser un error de digitacion.


In [35]:
print("Informacion hoja de Producto:")
print(f"\nMemoria usada: {df_products.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df_products.describe(include='all')

Informacion hoja de Producto:

Memoria usada: 2.82 MB


,ID_producto,Categoría,Subcategoría,Nombre_producto
count,10768,10768,10768,10768
unique,10292,3,17,3788
top,OFF-PA-10004673,Suministros_de_oficina,Papel,Staples
freq,4,5955,825,46


In [36]:
ids_repetidos = (
    df_products
    .groupby('ID_producto')['Nombre_producto']
    .nunique()
    .reset_index(name='nombres_distintos')
)

ids_conflictivos = ids_repetidos[ids_repetidos['nombres_distintos'] > 1]


In [37]:
ids_conflictivos['nombres_distintos'].value_counts()

nombres_distintos
2    439
3     17
4      1
Name: count, dtype: int64

In [38]:
detalle_conflicto = df_products[
    df_products['ID_producto'].isin(ids_conflictivos['ID_producto'])
][['ID_producto', 'Nombre_producto', 'Categoría', 'Subcategoría']]

detalle_conflicto.sort_values('ID_producto').head(10)

,ID_producto,Nombre_producto,Categoría,Subcategoría
56,FUR-BO-10000087,"Dania Classic Bookcase, Mobile",Muebles,Librerías
57,FUR-BO-10000087,"Sauder Corner Shelving, Pine",Muebles,Librerías
58,FUR-BO-10000112,"Dania Corner Shelving, Pine",Muebles,Librerías
59,FUR-BO-10000112,"Bush Birmingham Collection Bookcase, Dark Cherry",Muebles,Librerías
75,FUR-BO-10000268,"Bush Library with Doors, Pine",Muebles,Librerías
76,FUR-BO-10000268,"Ikea 3-Shelf Cabinet, Traditional",Muebles,Librerías
103,FUR-BO-10000668,"Sauder Classic Bookcase, Mobile",Muebles,Librerías
104,FUR-BO-10000668,"Ikea Corner Shelving, Traditional",Muebles,Librerías
112,FUR-BO-10000728,"Dania Corner Shelving, Traditional",Muebles,Librerías
113,FUR-BO-10000728,"Ikea Classic Bookcase, Mobile",Muebles,Librerías


#### NOTAS Hoja de Producto

Se identificaron **457** IDs de producto con diferentes nombres asociados (hasta 4 variantes por ID), lo que representa un error de diseño en el catalogo fuente.
Dado que la tabla de ordenes referencia unicamente el ID sin especificar la variante,
**se procedera** a consolidar estos registros bajo el primer nombre registrado de cada familia, conservando la Categoria y Subcategoria que son consistentes en 456 de 457 casos. Esta decision evita la generacion de filas duplicadas en un posible join con la hoja Ordenes.

---

**Recomendaciones:** Para los 457 IDs existe una limitacion, y es que se restringe a la identificación del nombre de producto exacto dentro de cada familia, la info. no esta disponible en el DS fuente. Se recomienda enriquecer el catalogo de productos con un identificador de variante **unico** por producto.


In [39]:
print("Informacion detallada de Region:")
print(f"\nMemoria usada: {df_regions.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df_regions.describe(include='all')

Informacion detallada de Region:

Memoria usada: 0.02 MB


,País,Mercado,Región
count,152,152,152
unique,147,7,13
top,United States,Africa,Africa
freq,4,45,45


In [40]:
df_regions['country_clean'] = (
    df_regions['País']
    .str.strip()
    .str.lower()
    .str.replace('.', '', regex=False)
)

df_regions['country_clean'].value_counts().reset_index(name='conteo')

,country_clean,conteo
0,united states,4
1,mongolia,2
2,austria,2
3,algeria,1
4,angola,1
...,...,...
142,ecuador,1
143,paraguay,1
144,peru,1
145,uruguay,1


#### NOTAS Hoja de Region

Se observa que existen paises repetidos con las siguientes incidencias:

* Mongolia - 2 registros: El registro A **ELIMINAR** es el Mongolia con con mercado y region **EMEA** ya que el otro registro tiene mercado APAC y region Asia, lo cual es mas consistente con la ubicacion real de Mongolia.

* Austria - 2 registros: El registro A **ELIMINAR** es el Austria con mercado y region **EMEA** ya que el otro registro tiene mercado EU y region Central, lo cual es mas consistente con la ubicacion real de Austria.

* United States - 4 registros: Los registros corresponden a las diferentes regiones de estados unidos (Central, East, South, West) por lo que se consolidara en una **unica fila** con Mercado = US y Regiòn = US.




===============================================================================================
# Limpieza transformacion de datos
===============================================================================================


#### Cambios para el DS Ordenes
* Imputacion de valores nulos en Nombre_del_cliente con "Cliente Desconocido"
* Eliminacion de registro con valores nulos en Ventas, Cantidad, Descuento, Ganancia, Costo_de_envío
* Correccion de error de digitacion en Pais "Colombia-" a "Colombia"
* Correccion de error de fecha del año 2034
* 1 Registro con Ventas = 0 [Se elimina por ser un error de digitacion y no aportar informacion relevante]
 

In [41]:
df_orders = df_orders.dropna(
    subset=["Ventas", "Cantidad", "Descuento", "Ganancia", "Costo_de_envío"],
    how="all"
)

df_orders["Nombre_del_cliente"] = (
    df_orders["Nombre_del_cliente"]
    .fillna("Cliente Desconocido")
)

df_orders["País"] = df_orders["País"].str.strip().replace("Colombia-", "Colombia")


mask_2034 = df_orders["Fecha_de_la_orden"].dt.year == 2034
df_orders.loc[mask_2034, "Fecha_de_la_orden"] = df_orders.loc[mask_2034, "Fecha_de_la_orden"].apply(
    lambda d: d.replace(year=2024))
df_orders.loc[mask_2034, "Fecha_de_envío"] = df_orders.loc[mask_2034, "Fecha_de_envío"].apply(
    lambda d: d.replace(year=2024))

df_orders = df_orders[
    ~(
        (df_orders["Ventas"] == 0) &
        (df_orders["Costo_de_envío"] > 0)
    )
]

print(f"Ordenes ok: {len(df_orders):,}")
print(f"Conteo de nulls: {df_orders.isnull().sum().sum()}")



Ordenes ok: 51,289
Conteo de nulls: 0


#### Cambios para el DS Producto
* Consolidacion de nombres de producto para los 457 IDs con multiples variantes, se mantiene la categoria y subcategoria del primer registro de cada familia.

In [42]:
# Consolidar IDs duplicados: 1 fila por ID_producto
# El Nombre_producto queda como el primer registro de cada familia
df_products_clean = (
    df_products
    .groupby("ID_producto", as_index=False)
    .agg(
        Categoría       = ("Categoría",       "first"),
        Subcategoría    = ("Subcategoría",    "first"),
        Nombre_producto = ("Nombre_producto", "first"),
    )
)

print(f"Filas producto: {len(df_products_clean):,} filas")
print(f"Duplicados resueltos: {len(df_products) - len(df_products_clean)}")

Filas producto: 10,292 filas
Duplicados resueltos: 476


#### Cambios para el DS Region
* Eliminacion de registros duplicados de Mongolia y Austria, se mantiene el registro con mercado y region mas consistente con la ubicacion geografica real de cada pais.

* Consolidacion de registros de United States en una unica fila con Mercado = US y Region = US.




In [43]:
mask_error = (
    ((df_regions["País"] == "Austria")  & (df_regions["Mercado"] == "EMEA")) |
    ((df_regions["País"] == "Mongolia") & (df_regions["Mercado"] == "EMEA"))
)
df_regions = df_regions[~mask_error].copy()

df_regions = df_regions[df_regions["País"] != "United States"].copy()
us_row = pd.DataFrame([{
    "País": "United States",
    "Mercado": "US",
    "Región": "US"
}])
df_regions = pd.concat([df_regions, us_row], ignore_index=True)


if "country_clean" in df_regions.columns:
    df_regions.drop(columns=["country_clean"], inplace=True)


print(df_regions[df_regions["País"].isin(["Austria", "Mongolia", "United States"])])


              País Mercado      Región
54        Mongolia    APAC  North Asia
107        Austria      EU     Central
146  United States      US          US


===============================================================================================
# Columnas adicionales
===============================================================================================


In [44]:
df_orders["Tiempo_entrega_dias"] = (df_orders["Fecha_de_envío"] - df_orders["Fecha_de_la_orden"]).dt.days
df_orders["Margen_pct"]          = (df_orders["Ganancia"] / df_orders["Ventas"] * 100).round(2)
df_orders["Es_rentable"]         = (df_orders["Ganancia"] > 0).astype(int)
df_orders["Tiene_descuento"]     = (df_orders["Descuento"] > 0).astype(int)
df_orders["Año"]                 = df_orders["Fecha_de_la_orden"].dt.year
df_orders["Trimestre"]           = df_orders["Fecha_de_la_orden"].dt.quarter
df_orders["Mes"]                 = df_orders["Fecha_de_la_orden"].dt.month
df_orders["Nombre_mes"]          = df_orders["Fecha_de_la_orden"].dt.strftime("%b")
df_orders["AñoMes"]              = df_orders["Fecha_de_la_orden"].dt.to_period("M").astype(str)
df_orders["Segmento_descuento"]  = pd.cut(df_orders["Descuento"],
    bins=[-0.01,0,0.2,0.4,1.0],
    labels=["Sin descuento","Bajo (1-20%)","Medio (21-40%)","Alto (>40%)"])
df_orders["Cat_entrega"] = pd.cut(df_orders["Tiempo_entrega_dias"],
    bins=[-1,2,4,7,999],
    labels=["Rápido (≤2d)","Normal (3-4d)","Lento (5-7d)","Muy lento (>7d)"])
 
# — Join maestro
df = (df_orders
      .merge(df_products_clean, on="ID_producto", how="left")
      .merge(df_regions,        on="País",        how="left"))
 
print(f" Dataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
 

✅ Dataset final: 51,289 filas × 30 columnas


### Preparacion Nuevo DS listo apra analisis y modelado
---


In [45]:
#DS limpio
cols_export = [
    "ID_orden","ID_producto","Fecha_de_la_orden","Fecha_de_envío",
    "Año","Trimestre","Mes","Nombre_mes","AñoMes",
    "Nombre_del_cliente","Estado_Departamento","País","Región","Mercado",
    "Método_de_envío","Prioridad_de_orden","Tiempo_entrega_dias","Cat_entrega",
    "Categoría","Subcategoría","Nombre_producto",
    "Cantidad","Ventas","Descuento","Segmento_descuento","Tiene_descuento",
    "Ganancia","Margen_pct","Es_rentable","Costo_de_envío",
]
OUT = "../outputs"   
os.makedirs(OUT, exist_ok=True)
df[cols_export].to_excel(f"{OUT}/dataset_limpio_BI.xlsx", index=False,
                          sheet_name="Dataset_Limpio")
print(f"Dataset exportado nombre del archivo: dataset_limpio_BI.xlsx")

💾 Dataset exportado → dataset_limpio_BI.xlsx


===============================================================================================
# ESTADISTICAS DESCRIPTIVAS Y ANALISIS
===============================================================================================
## Resumen Ejecutivo de Datos

In [52]:
#PREGUNTAS CLAVE 

#¿Cuantas Ordenes hay y PerIodo
total_ordenes = df["ID_orden"].nunique()
fecha_min = df["Fecha_de_la_orden"].min()
fecha_max = df["Fecha_de_la_orden"].max()
periodo_dias = (fecha_max - fecha_min).days

print(f"\n ORDENES Y PERIODO:")
print(f"   Total de Ordenes Unicas: {total_ordenes:,}")
print(f"   Periodo: {fecha_min.strftime('%Y-%m-%d')} a {fecha_max.strftime('%Y-%m-%d')}")
print(f"   Duracion: {periodo_dias} dias (~{periodo_dias//365} años)")

#¿Cuantos clientes unicos hay?
clientes_unicos = df["Nombre_del_cliente"].nunique()
print(f"\n CLIENTES:")
print(f"   Total de clientes unicos: {clientes_unicos:,}")
print(f"   Promedio de ordenes por cliente: {total_ordenes / clientes_unicos:.1f}")

#¿Cual es el monto promedio de orden?
total_ventas = df["Ventas"].sum()
monto_promedio_orden = df.groupby("ID_orden")["Ventas"].sum().mean()

print(f"\n MONTO DE ORDEN:")
print(f"   Total de ventas: ${total_ventas:,.2f}")
print(f"   Monto promedio por orden: ${monto_promedio_orden:,.2f}")

#Productos unicos?
productos_unicos = df["ID_producto"].nunique()
print(f"\n PRODUCTOS:")
print(f"   Total de productos unicos: {productos_unicos:,}")
print(f"   Categorias unicas: {df['Categoría'].nunique()}")
print(f"   Subcategorías unicas: {df['Subcategoría'].nunique()}")

#TOP regiones lideran
print(f"\n TOP REGIONES (por ventas):")
top_paises = df.groupby("País")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_paises.columns = ["Ventas_Total", "Num_Ordenes"]
for idx, (pais, row) in enumerate(top_paises.head(5).iterrows(), 1):
    pct_ventas = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {pais}: ${row['Ventas_Total']:,.0f} ({pct_ventas:.1f}%) - {int(row['Num_Ordenes'])} órdenes")

#Productos top
print(f"\n TOP 10 PRODUCTOS (por ventas):")
top_productos = df.groupby("Nombre_producto")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_productos.columns = ["Ventas_Total", "Cantidad_Vendida"]
for idx, (prod, row) in enumerate(top_productos.head(10).iterrows(), 1):
    pct_ventas = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {prod}: ${row['Ventas_Total']:,.0f} ({pct_ventas:.1f}%) - {int(row['Cantidad_Vendida'])} unidades")

#Productos TOP por cantidad
print(f"\n TOP 10 PRODUCTOS (por cantidad vendida):")
top_productos_cant = df.groupby("Nombre_producto")["Cantidad"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_productos_cant.columns = ["Cantidad_Total", "Num_Ordenes"]
for idx, (prod, row) in enumerate(top_productos_cant.head(10).iterrows(), 1):
    print(f"   {idx}. {prod}: {int(row['Cantidad_Total']):,} unidades - {int(row['Num_Ordenes'])} órdenes")

#TOP Categorias mas vendidas por cantidad
print(f"\n TOP CATEGORIAS (por cantidad vendida):")
top_categorias_cant = df.groupby("Categoría")["Cantidad"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_categorias_cant.columns = ["Cantidad_Total", "Num_Ordenes"]
for idx, (cat, row) in enumerate(top_categorias_cant.head(5).iterrows(), 1):
    print(f"   {idx}. {cat}: {int(row['Cantidad_Total']):,} unidades - {int(row['Num_Ordenes'])} órdenes")



 ORDENES Y PERIODO:
   Total de Ordenes Unicas: 25,034
   Periodo: 2021-01-01 a 2024-12-31
   Duracion: 1460 dias (~4 años)

 CLIENTES:
   Total de clientes unicos: 796
   Promedio de ordenes por cliente: 31.4

 MONTO DE ORDEN:
   Total de ventas: $12,642,452.41
   Monto promedio por orden: $505.01

 PRODUCTOS:
   Total de productos unicos: 10,292
   Categorias unicas: 3
   Subcategorías unicas: 17

 TOP REGIONES (por ventas):
   1. United States: $2,297,201 (18.2%) - 9994 órdenes
   2. Australia: $925,236 (7.3%) - 2837 órdenes
   3. France: $858,931 (6.8%) - 2827 órdenes
   4. China: $700,562 (5.5%) - 1880 órdenes
   5. Germany: $628,840 (5.0%) - 2065 órdenes

 TOP 10 PRODUCTOS (por ventas):
   1. Apple Smart Phone, Full Size: $86,936 (0.7%) - 51 unidades
   2. Cisco Smart Phone, Full Size: $76,442 (0.6%) - 38 unidades
   3. Motorola Smart Phone, Full Size: $73,156 (0.6%) - 38 unidades
   4. Nokia Smart Phone, Full Size: $71,905 (0.6%) - 47 unidades
   5. Canon imageCLASS 2200 Advanc

---

## Estadísticas Detalladas de Ventas y Rentabilidad

In [56]:
#ANALISIS DE VENTAS Y RENTABILIDAD

print("\n" + "=" * 80)
print("ANALISIS DE VENTAS Y RENTABILIDAD")
print("=" * 80)

#Analisis de ganancia
total_ganancia = df["Ganancia"].sum()
ganancia_promedio = df["Ganancia"].mean()
margen_promedio = df["Margen_pct"].mean()
ordenes_rentables = df.groupby("ID_orden")["Es_rentable"].max().sum()
pct_rentables = (ordenes_rentables / total_ordenes) * 100

print(f"\n RENTABILIDAD:")
print(f"   Ganancia total: ${total_ganancia:,.2f}")
print(f"   Ganancia promedio por linea: ${ganancia_promedio:,.2f}")
print(f"   Margen promedio: {margen_promedio:.2f}%")
print(f"   Ordenes rentables: {ordenes_rentables:,} ({pct_rentables:.1f}%)")
print(f"   Ordenes con perdida: {total_ordenes - ordenes_rentables:,} ({100-pct_rentables:.1f}%)")

#AnAlisis de descuentos
ordenes_con_descuento = df[df["Descuento"] > 0]["ID_orden"].nunique()
pct_descuento = (ordenes_con_descuento / total_ordenes) * 100
descuento_promedio = df[df["Descuento"] > 0]["Descuento"].mean() * 100
impacto_descuento = df[df["Descuento"] > 0]["Ganancia"].sum()

print(f"\n DESCUENTOS:")
print(f"   Ordenes con descuento: {ordenes_con_descuento:,} ({pct_descuento:.1f}%)")
print(f"   Descuento promedio (cuando aplica): {descuento_promedio:.2f}%")
print(f"   Ganancia de ordenes con descuento: ${impacto_descuento:,.2f}")

#Analisis de entregas
tiempo_entrega_promedio = df["Tiempo_entrega_dias"].mean()
print(f"\n ENTREGAS:")
print(f"   Tiempo de entrega promedio: {tiempo_entrega_promedio:.1f} dias")
print(f"   Minimo: {df['Tiempo_entrega_dias'].min()} dias")
print(f"   Maximo: {df['Tiempo_entrega_dias'].max()} dias")
print(f"   Categoria de entrega mas comun: {df['Cat_entrega'].mode()[0]}")

#Analisis por categoraa
print(f"\n  VENTAS POR CATEGORiA:")
top_categorias = df.groupby("Categoría")["Ventas"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)
top_categorias.columns = ["Ventas_Total", "Items", "Ventas_Promedio"]
for idx, (cat, row) in enumerate(top_categorias.head(5).iterrows(), 1):
    pct = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {cat}: ${row['Ventas_Total']:,.0f} ({pct:.1f}%) - {int(row['Items'])} items")


ANALISIS DE VENTAS Y RENTABILIDAD

 RENTABILIDAD:
   Ganancia total: $1,469,027.98
   Ganancia promedio por linea: $28.64
   Margen promedio: 4.74%
   Ordenes rentables: 20,134 (80.4%)
   Ordenes con perdida: 4,900 (19.6%)

 DESCUENTOS:
   Ordenes con descuento: 12,152 (48.5%)
   Descuento promedio (cuando aplica): 32.90%
   Ganancia de ordenes con descuento: $-301,660.45

 ENTREGAS:
   Tiempo de entrega promedio: 4.0 dias
   Minimo: 0 dias
   Maximo: 8 dias
   Categoria de entrega mas comun: Lento (5-7d)

  VENTAS POR CATEGORiA:
   1. Tecnología: $4,744,557 (37.5%) - 10141 items
   2. Muebles: $4,110,874 (32.5%) - 9876 items
   3. Suministros_de_oficina: $3,787,021 (30.0%) - 31272 items


---


In [61]:
#Distribucion por mercado
print(f"\n🌐 VENTAS POR MERCADO:")
mercados = df.groupby("Mercado")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
mercados.columns = ["Ventas", "Num_Items"]
for mercado, row in mercados.iterrows():
    pct = (row["Ventas"] / total_ventas) * 100
    print(f"   {mercado}: ${row['Ventas']:,.0f} ({pct:.1f}%) - {int(row['Num_Items'])} items")

#Clientes top por valor
print(f"\n TOP 10 CLIENTES (por valor gastado):")
clientes_valor = df.groupby("Nombre_del_cliente")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
clientes_valor.columns = ["Ventas_Total", "Num_Ordenes"]
for idx, (cliente, row) in enumerate(clientes_valor.head(10).iterrows(), 1):
    pct = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {cliente}: ${row['Ventas_Total']:,.0f} ({pct:.1f}%) - {int(row['Num_Ordenes'])} órdenes")

#Clientes top por cantidad
print(f"\n TOP 10 CLIENTES (por cantidad comprada):")
clientes_cantidad = df.groupby("Nombre_del_cliente")["Cantidad"].agg(["sum", "count"]).sort_values("sum", ascending=False)
clientes_cantidad.columns = ["Cantidad_Total", "Num_Ordenes"]
for idx, (cliente, row) in enumerate(clientes_cantidad.head(10).iterrows(), 1):
    print(f"   {idx}. {cliente}: {int(row['Cantidad_Total']):,} unidades - {int(row['Num_Ordenes'])} órdenes")


🌐 VENTAS POR MERCADO:
   APAC: $3,592,494 (28.4%) - 11038 items
   EU: $2,949,466 (23.3%) - 10061 items
   US: $2,297,201 (18.2%) - 9994 items
   LATAM: $2,164,605 (17.1%) - 10294 items
   EMEA: $788,035 (6.2%) - 4932 items
   Africa: $783,724 (6.2%) - 4586 items
   Canada: $66,928 (0.5%) - 384 items

 TOP 10 CLIENTES (por valor gastado):
   1. Tom Ashbrook: $40,488 (0.3%) - 80 órdenes
   2. Tamara Chand: $37,457 (0.3%) - 88 órdenes
   3. Greg Tran: $35,551 (0.3%) - 87 órdenes
   4. Christopher Conant: $35,187 (0.3%) - 73 órdenes
   5. Sean Miller: $35,171 (0.3%) - 50 órdenes
   6. Bart Watters: $32,310 (0.3%) - 96 órdenes
   7. Natalie Fritzler: $31,781 (0.3%) - 95 órdenes
   8. Fred Hopkins: $30,401 (0.2%) - 82 órdenes
   9. Jane Waco: $30,288 (0.2%) - 75 órdenes
   10. Hunter Lopez: $30,244 (0.2%) - 53 órdenes

 TOP 10 CLIENTES (por cantidad comprada):
   1. Bill Eplett: 411 unidades - 102 órdenes
   2. Eric Murdock: 392 unidades - 100 órdenes
   3. Steven Ward: 383 unidades - 106 

---

## Visualizaciones Clave

In [63]:
#Grafico de Ventas por País (Top 10)
fig1 = px.bar(
    top_paises.reset_index().head(10),
    x='País',
    y='Ventas_Total',
    title='Top 10 Países por Ventas Totales',
    labels={'Ventas_Total': 'Ventas ($)', 'País': 'País'},
    color='Ventas_Total',
    color_continuous_scale='Viridis'
)
fig1.show()

#Gráfico de Top Productos
fig2 = px.bar(
    top_productos.reset_index().head(10),
    x='Nombre_producto',
    y='Ventas_Total',
    title='Top 10 Productos por Ventas',
    labels={'Ventas_Total': 'Ventas ($)', 'Nombre_producto': 'Producto'},
    color='Ventas_Total',
    color_continuous_scale='Blues'
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

#Ventas por Mes (Tendencia)
ventas_mes = df.groupby("AñoMes")["Ventas"].sum().reset_index()
fig3 = px.line(
    ventas_mes,
    x='AñoMes',
    y='Ventas',
    title='Tendencia de Ventas Mensuales',
    markers=True,
    labels={'Ventas': 'Ventas ($)', 'AñoMes': 'Año-Mes'}
)
fig3.show()


#Pie Chart: Ventas por Mercado
fig5 = px.pie(
    df.groupby("Mercado")["Ventas"].sum().reset_index(),
    values='Ventas',
    names='Mercado',
    title='Distribución de Ventas por Mercado'
)
fig5.show()

#Scatter: Ventas vs Ganancia (por línea)
fig6 = px.scatter(
    df.head(1000),  # Limitando a 1000 para claridad visual
    x='Ventas',
    y='Ganancia',
    color='Margen_pct',
    size='Cantidad',
    title='Relación Ventas vs Ganancia (muestreo)',
    labels={'Ventas': 'Ventas ($)', 'Ganancia': 'Ganancia ($)', 'Margen_pct': 'Margen (%)'},
    hover_data=['Nombre_producto', 'Margen_pct']
)
fig6.show()

---



In [68]:
# Heatmap Categoría vs Subcategoría
print("\n📈 VENTAS POR SUBCATEGORÍA (top 3 por categoría):")
for cat in df['Categoría'].unique():
    print(f"\n{cat}:")
    subcat = df[df['Categoría'] == cat].groupby("Subcategoría")["Ventas"].sum().sort_values(ascending=False).head(3)
    for subcat_name, ventas in subcat.items():
        pct = (ventas / total_ventas) * 100
        print(f"   - {subcat_name}: ${ventas:,.0f} ({pct:.2f}%)")

# Gráfico: Categorías vs Ventas
fig7 = px.bar(
    categoria_totales.reset_index().sort_values("Ventas", ascending=False),
    x='Categoría',
    y='Ventas',
    title='Ventas Totales por Categoría',
    labels={'Ventas': 'Ventas ($)', 'Categoría': 'Categoría'},
    color='Margen_pct',
    color_continuous_scale='RdYlGn',
    text='Ganancia'
)
fig7.show()

# Gráfico: Rentabilidad por Categoría
rentabilidad_cat = df.groupby("Categoría").agg({
    "Ganancia": "sum",
    "Ventas": "sum",
    "Es_rentable": "mean"
}).sort_values("Ganancia", ascending=False)

fig8 = px.bar(
    rentabilidad_cat.reset_index(),
    x='Categoría',
    y='Ganancia',
    title='Ganancia Total por Categoría',
    color='Es_rentable',
    labels={'Ganancia': 'Ganancia ($)', 'Es_rentable': 'Tasa Rentabilidad'}
)
fig8.show()


📈 VENTAS POR SUBCATEGORÍA (top 3 por categoría):

Suministros_de_oficina:
   - Almacenamiento: $1,127,086 (8.92%)
   - Electrodomésticos: $1,011,064 (8.00%)
   - Carpetas: $461,931 (3.65%)

Muebles:
   - Sillas: $1,501,682 (11.88%)
   - Librerías: $1,466,572 (11.60%)
   - Mesas: $757,042 (5.99%)

Tecnología:
   - Teléfonos: $1,706,824 (13.50%)
   - Fotocopiadoras: $1,509,436 (11.94%)
   - Máquinas: $779,060 (6.16%)


---

In [70]:
# ========== RESUMEN FINAL Y HALLAZGOS CLAVE ==========

print("\n" + "=" * 80)
print("HALLAZGOS CLAVE DEL EDA")
print("=" * 80)

hallazgos = f"""

✅ HALLAZGOS PRINCIPALES:

1. 📊 VOLUMEN DE NEGOCIO
   • {total_ordenes:,} órdenes únicas en el período de {(fecha_max - fecha_min).days} días
   • {clientes_unicos:,} clientes únicos ({total_ordenes / clientes_unicos:.1f} órdenes/cliente en promedio)
   • {productos_unicos} productos vendidos

2. 💰 INGRESOS Y RENTABILIDAD
   • Ingresos totales: ${total_ventas:,.0f}
   • Ganancia total: ${total_ganancia:,.0f} (margen global: {(total_ganancia/total_ventas*100):.2f}%)
   • {ordenes_rentables:,} órdenes rentables ({pct_rentables:.1f}%) vs {total_ordenes - ordenes_rentables:,} con pérdida
   • Monto promedio de orden: ${monto_promedio_orden:,.0f}

3. 🌍 CONCENTRACIÓN GEOGRÁFICA
   • Top 3 países: {', '.join(top_paises.index[:3].tolist())}
   • Los 5 principales países representan el {(top_paises['Ventas_Total'].head(5).sum()/total_ventas*100):.1f}% de ventas
   • {len(df['País'].unique())} países en total en el portafolio

4. 📦 ANÁLISIS DE PRODUCTOS
   • {productos_unicos} productos únicos / {df['Categoría'].nunique()} categorías / {df['Subcategoría'].nunique()} subcategorías
   • Top producto: {top_productos.index[0]} (${top_productos.iloc[0]['Ventas_Total']:,.0f})
   • Concentración: Top 10 productos = {(top_productos.head(10)['Ventas_Total'].sum()/total_ventas*100):.1f}% de ventas

5. 🎟️ DESCUENTOS E IMPACTO
   • {ordenes_con_descuento:,} órdenes con descuento ({pct_descuento:.1f}%)
   • Descuento promedio (cuando aplica): {descuento_promedio:.2f}%
   • Impacto en ganancia: ${impacto_descuento:,.0f}

6. 👥 SEGMENTACIÓN DE CLIENTES
   • VIP (frecuencia alta + valor alto): {(rfm['Segmento']=='VIP').sum()} clientes
   • Regular: {(rfm['Segmento']=='Regular').sum()} clientes
   • At-Risk (sin compras recientes): {(rfm['Segmento']=='At-Risk').sum()} clientes

7. 📈 ANÁLISIS PARETO
   • {len(clientes_80pct)} clientes ({len(clientes_80pct)/clientes_unicos*100:.1f}%) generan 80% de ventas
   • Ratio 80/20: {len(clientes_80pct)/clientes_unicos:.1f}x superior a lo esperado

8. 🚚 OPERACIONES DE ENTREGA
   • Tiempo promedio de entrega: {tiempo_entrega_promedio:.1f} días
   • Rango: {df['Tiempo_entrega_dias'].min()} a {df['Tiempo_entrega_dias'].max()} días

========================================================================================
PRÓXIMOS PASOS RECOMENDADOS:
========================================================================================

✓ El EDA está COMPLETO y LISTO para Power BI
✓ Datos limpios exportados a: dataset_limpio_BI.xlsx
✓ Todas las preguntas clave RESPONDIDAS

ACCIONES INMEDIATAS:
1. Exportar datos adicionales por segmento para Power BI
2. Crear tabla de RFM para visualización
3. Preparar datos de tendencias mensuales
4. Diseñar dashboard en Power BI basado en estos análisis
"""

print(hallazgos)


HALLAZGOS CLAVE DEL EDA


✅ HALLAZGOS PRINCIPALES:

1. 📊 VOLUMEN DE NEGOCIO
   • 25,034 órdenes únicas en el período de 1460 días
   • 796 clientes únicos (31.4 órdenes/cliente en promedio)
   • 10292 productos vendidos

2. 💰 INGRESOS Y RENTABILIDAD
   • Ingresos totales: $12,642,452
   • Ganancia total: $1,469,028 (margen global: 11.62%)
   • 20,134 órdenes rentables (80.4%) vs 4,900 con pérdida
   • Monto promedio de orden: $505

3. 🌍 CONCENTRACIÓN GEOGRÁFICA
   • Top 3 países: United States, Australia, France
   • Los 5 principales países representan el 42.8% de ventas
   • 147 países en total en el portafolio

4. 📦 ANÁLISIS DE PRODUCTOS
   • 10292 productos únicos / 3 categorías / 17 subcategorías
   • Top producto: Apple Smart Phone, Full Size ($86,936)
   • Concentración: Top 10 productos = 5.0% de ventas

5. 🎟️ DESCUENTOS E IMPACTO
   • 12,152 órdenes con descuento (48.5%)
   • Descuento promedio (cuando aplica): 32.90%
   • Impacto en ganancia: $-301,660

6. 👥 SEGMENTACIÓN DE 